# Step 9 — Corrected bias metrics and a reduced 1-D prediction of generated class fraction

**Fixes.** (1) The class-fraction baseline is the *real balanced data's* `frac(m0 > 0)` along LDA, not 0.5. (2) Commitment τ uses the LDA coordinate of the **denoised** estimate `D(x_t)`, not the noisy state.

**Prediction.** The network's binned drift `g(m; σ) = E[score_m | m]` defines a reduced 1-D probability-flow ODE, `dm/dσ = −σ g(m;σ)/half²`. Integrating it from `σ_max` for many initial `m` gives a predicted generated fraction. Compare with the actual 2,000-sample fraction per checkpoint. Agreement means the 1-D reduction is a predictive model of commitment; disagreement localizes where it breaks.

Mount the Step 7 notebook output (checkpoints). GPU on, ~12 min.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os
import matplotlib.pyplot as plt
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 110
ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X_ALL = (ds.data.numpy().astype(np.float32)/127.5 - 1.0); Y_ALL = ds.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]; SIGMA_DATA = 0.5; CKPTS = [20, 40, 60, 80]
found = glob.glob('/kaggle/input/**/unet_*_ep*.pt', recursive=True); assert found, 'mount the Step 7 notebook output'
CKPT_DIR = os.path.dirname(found[0]); print(CKPT_DIR)

class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def load_nets(pair):
    out = {}
    for ep in CKPTS:
        net = UNet().to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/unet_{pair[0]}{pair[1]}_ep{ep}.pt', map_location=dev)); out[ep] = net.eval()
    return out
NETS = {pair: load_nets(pair) for pair in PAIRS}

def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced_flat(pair):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); return Xa, Xb, np.concatenate([Xa[:nmin], Xb[:nmin]])

## Sampler with denoised trajectory, and the reduced 1-D ODE

In [ ]:
S_MAX, S_MIN, STEPS, RHO = 20.0, 0.002, 40, 7.0
def sigma_schedule():
    i = torch.arange(STEPS, device=dev); sig = (S_MAX**(1/RHO) + i/(STEPS-1)*(S_MIN**(1/RHO) - S_MAX**(1/RHO)))**RHO
    return torch.cat([sig, torch.zeros(1, device=dev)])

@torch.no_grad()
def sample(net, ut, midt, half, n=2000):
    """Heun PF-ODE sampler. Returns final x (n,784), sigmas, and the LDA coordinate of the DENOISED estimate at each step (steps, n)."""
    sig = sigma_schedule(); x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]; mhat = []
    for k in range(STEPS):
        s0, s1 = sig[k], sig[k+1]; D0 = denoise(net, x, s0.expand(n)); mhat.append(((D0.view(n, -1) - midt) @ ut / half).clone())
        d0 = (x - D0)/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - denoise(net, x1, s1.expand(n)))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return x.view(n, -1), sig[:-1].cpu().numpy(), torch.stack(mhat).cpu().numpy()

@torch.no_grad()
def drift_table(net, Xflat, u, mid, half, sigmas, n=20000, nbins=80):
    """Binned network drift g(m; sigma) = E[score_m | m] on forward-noised data, for each sigma. Returns centers, table (len(sigmas), nbins)."""
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); table = []; edges_all = []
    for sigma in sigmas:
        x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
        D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), float(sigma), device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
        m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
        lo, hi = np.quantile(m, [0.0005, 0.9995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
        cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
        g = np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
        table.append((0.5*(edges[:-1]+edges[1:]), g))
    return table

def reduced_ode_fraction(table, sigmas, half, m0_mean, n=20000):
    """Integrate dm/dsigma = -sigma * g(m; sigma)/half^2 (Heun) from sigma_max, with m ~ N(m0_mean, sigma_max^2/half^2). Return frac(m_final > 0)."""
    def g(m, k):
        c, gg = table[k]; return np.interp(m, c, gg)
    m = m0_mean + rng.standard_normal(n)*sigmas[0]/half
    for k in range(len(sigmas)-1):
        s0, s1 = sigmas[k], sigmas[k+1]
        d0 = -s0*g(m, k)/half**2; m1 = m + (s1-s0)*d0
        d1 = -s1*g(m1, k+1)/half**2; m = m + (s1-s0)*0.5*(d0+d1)
    return float((m > 0).mean()), m

## Run: baseline, generated fraction, commitment τ, reduced-ODE prediction

In [ ]:
sig_np = sigma_schedule()[:-1].cpu().numpy()
print(f"{'pair':>7} {'baseline':>9} | {'ep':>4} {'gen frac':>9} {'excess':>8} {'1D-ODE frac':>12} {'1D excess':>10} {'commit tau med':>15}")
RES = []
for pair in PAIRS:
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb)
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    m0 = (Xflat-mid) @ u / half; baseline = float((m0 > 0).mean()); m0_mean = float(-(mid @ u)/half)   # coordinate of x=0 (mean of the initial noise)
    for ep in CKPTS:
        net = NETS[pair][ep]
        x, sigs, mhat = sample(net, ut, midt, half); gen = float((((x - midt) @ ut / half) > 0).float().mean())
        committed = np.abs(mhat) > 0.5; first = np.array([np.argmax(c) if c.any() else STEPS-1 for c in committed.T]); tau_c = np.median(sigs[first]**2/half**2)
        table = drift_table(net, Xflat, u, mid, half, sig_np); pred, mfin = reduced_ode_fraction(table, sig_np, half, m0_mean)
        RES.append(dict(pair=pair, ep=ep, baseline=baseline, gen=gen, pred=pred, tau_c=tau_c))
        print(f"{pair!s:>7} {baseline:9.3f} | {ep:4d} {gen:9.3f} {gen-baseline:+8.3f} {pred:12.3f} {pred-baseline:+10.3f} {tau_c:15.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for pair, col in zip(PAIRS, ['C0', 'C1', 'C2']):
    r = [x for x in RES if x['pair'] == pair]
    axes[0].plot([x['ep'] for x in r], [x['gen']-x['baseline'] for x in r], 'o-', color=col, label=f'{pair} sampled')
    axes[0].plot([x['ep'] for x in r], [x['pred']-x['baseline'] for x in r], 's--', color=col, alpha=0.7, label=f'{pair} 1-D ODE')
    axes[1].scatter([x['pred'] for x in r], [x['gen'] for x in r], color=col, label=str(pair))
axes[0].axhline(0, color='k', lw=0.6); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('excess + fraction over real-data baseline'); axes[0].legend(fontsize=7)
lim = [0.4, 0.8]; axes[1].plot(lim, lim, 'k--', lw=0.8); axes[1].set_xlabel('1-D reduced ODE prediction'); axes[1].set_ylabel('sampled fraction'); axes[1].legend(); axes[1].set_title('does the 1-D reduction predict the bias?')
plt.tight_layout(); plt.show()

## What to report back

1. The table: real-data baseline per pair, the **excess** generated fraction per checkpoint, and the reduced-ODE prediction.
2. The right panel: if points lie near the diagonal, the 1-D reduction along LDA predicts class bias from the drift alone. Systematic deviation (e.g. the ODE overpredicting the bias) indicates where the generated marginal departs from the forward-noised marginal the drift was measured on.
3. Commitment τ (now meaningful): should be O(1) and move toward the exact fold τ (~0.8–0.9) with training.